# Base Tools und Functions for Neighbor Analysis and Image Augmentation

In [ ]:
from IPython.utils.capture import capture_output

# Import data from the base notebook

# Use "capture" do suppress plotting images
with capture_output() as captured:
    %run r0_dataset_overiew_dashboard.ipynb

In [2]:
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
import umap
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
from collections import Counter
import pandas as pd
%matplotlib inline

In [3]:
dataset = XRAY_DATASET
#images = np.stack([x["img"] for x in XRAY_DATASET])

In [4]:
print(len(dataset))
print(MAX_NUMBER_OF_DATA)

10000
10000


In [5]:
def query_error_with_downprojection(images, error_idx = None, query_idx: int = None, n_features: int = 100, n_neighbors: int = 10, seed=42):

    if query_idx == None:
        if error_idx is None:
            raise ValueError("query_idx or error_idx for FP/TP/FN/TN required!")
        query_idx = np.random.default_rng(seed).choice(error_idx) # random selection

    # Flatten images for pixel-based comparison
    X_raw = images.reshape(len(images), -1).astype(np.float32)

    # PCA
    pca = PCA(n_components=n_features)
    X_pca = pca.fit_transform(X_raw)       

    # UMAP (with PCA features)
    X_umap = umap.UMAP(n_components=2,
                              n_neighbors=n_neighbors,
                              min_dist=0.1,
                              random_state=seed).fit_transform(X_pca)

    # NN
    feature_space = {"Raw": X_raw,
                     "PCA": X_pca,
                     "UMAP": X_umap}

    neighbors = {}

    # TODO: t-SNE OR use downprojector from other notebook

    for space, features in feature_space.items():
        nn = NearestNeighbors(n_neighbors=n_neighbors + 1, metric="euclidean")
        nn.fit(features)
        distances, indices = nn.kneighbors(features[query_idx:query_idx + 1])

        # Exclude query image
        mask = indices[0] != query_idx
        neighbor_indices = indices[0][mask][:n_neighbors]
        neighbor_distances = distances[0][mask][:n_neighbors]
        # Query + nearest neighbors
        plot_indices = [query_idx] + list(neighbor_indices)

        neighbors[space] = neighbor_indices, neighbor_distances, plot_indices

    return query_idx, neighbors, feature_space


In [6]:
def neighbor_overlap(data_1, data_2):
    indices_1, _, _ = data_1
    indices_2, _, _ = data_2

    set_1 = set(np.asarray(indices_1).ravel())
    set_2 = set(np.asarray(indices_2).ravel())

    return len(set_1 & set_2) / len(set_1)

def calculate_overlap_matrix(neighbors: dict):
    methods = list(neighbors)

    return np.array([[neighbor_overlap(neighbors[a], neighbors[b]) for b in methods] for a in methods])

def visualize_overlap(neighbors: dict):
    methods = list(neighbors)

    fig, ax = plt.subplots(figsize=(5, 4))

    sns.heatmap(calculate_overlap_matrix(neighbors), 
                ax=ax,
                annot=True,
                fmt=".0%",
                xticklabels=methods,
                yticklabels=methods,
                cmap="Blues",
                vmin=0,
                vmax=1,
                square=True)

    ax.set_title("Shared Neighbors")
    fig.tight_layout()
    plt.show()

In [7]:
def plot_neighbor_comparison(query_idx, neighbors, images, ground_truth, predictions, threshold=0.5):

    border_colors = {3: "mediumseagreen",
                     2: "orange",
                     1: "lightgray"}

    # Count neighbor overlap across methods (query excluded)
    counts = Counter(
        int(idx)
        for neighbor_indices, _, _ in neighbors.values()
        for idx in set(neighbor_indices)
        if idx != query_idx)

    n_cols = max(len(data[2]) for data in neighbors.values())

    fig, axes = plt.subplots(
        len(neighbors),
        n_cols,
        figsize=(2.4 * n_cols, 3 * len(neighbors)),
        squeeze=False)

    for row, (method, (neighbor_indices, neighbor_distances, plot_indices)) in enumerate(neighbors.items()):

        for col, img_idx in enumerate(plot_indices):

            ax = axes[row, col]
            img_idx = int(img_idx)

            ax.imshow(images[img_idx].squeeze(), cmap="gray")

            classification = classification = ("TP" if ground_truth[img_idx] == 1 and predictions[img_idx] >= threshold else
                                               "FN" if ground_truth[img_idx] == 1 else
                                               "FP" if predictions[img_idx] >= threshold else
                                               "TN")
            gt = ground_truth[img_idx]
            pred = predictions[img_idx]

            if col == 0:
                color = "red"
                title = (
                    f"Query ({classification})\n"
                    f"GT: {gt:.0f} | Pred: {pred:.3f}\n"
                    f"Idx: {img_idx}")
            else:
                color = border_colors[counts[img_idx]]
                title = (
                    f"Neighbor {col} ({classification})\n"
                    f"Dist: {neighbor_distances[col - 1]:,.2f}\n"
                    f"GT: {gt:.0f} | Pred: {pred:.3f}\n"
                    f"Idx: {img_idx}")

            ax.set_title(title, fontsize=9)

            for spine in ax.spines.values():
                spine.set_visible(True)
                spine.set_edgecolor(color)
                spine.set_linewidth(3)

            ax.set_xticks([])
            ax.set_yticks([])

            if col == 0:
                ax.set_ylabel(method, fontsize=12, labelpad=15)

        # Hide unused axes if methods have different neighbor counts
        for col in range(len(plot_indices), n_cols):
            axes[row, col].axis("off")

    #plt.tight_layout()
    #plt.show()

    legend_elements = [Patch(facecolor="red", label="Query"),
                        Patch(facecolor="mediumseagreen", label="Shared by 3 methods"),
                        Patch(facecolor="orange", label="Shared by 2 methods"),
                        Patch(facecolor="lightgray", label="Unique to 1 method")]

    fig.legend(
        handles=legend_elements,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.01),
        ncol=4,
        frameon=False,
        fontsize=10)

    fig.suptitle("Nearest Neighbor Comparison")

    fig.subplots_adjust(
    top=0.85,      # Space for main title
    bottom=0.10,   # Space for legend
    hspace=0.65,   # Vertical spacing between rows
    wspace=0.12)   # Horizontal spacing between images

    plt.show()

In [8]:
def plot_neighborhood_projections(query_idx, neighbors, feature_space, ground_truth, k=10, show_projected_neighbors=True):
    # Use the first two PCA components
    projections = {
        "PCA": feature_space["PCA"][:, :2],
        "UMAP": feature_space["UMAP"]}

    # Raw-pixel neighbors from your existing calculation
    raw_neighbors = np.asarray(neighbors["Raw"][2])[:k]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    for ax, (method, X_2d) in zip(axes, projections.items()):

        # Dataset, colored by ground truth
        scatter = ax.scatter(X_2d[:, 0],
                            X_2d[:, 1],
                            c=ground_truth,
                            cmap="coolwarm",
                            vmin=0,
                            vmax=1,
                            s=12,
                            alpha=0.5)

        # Highlight original raw-pixel neighbors
        ax.scatter(X_2d[raw_neighbors, 0],
                   X_2d[raw_neighbors, 1],
                   s=110,
                   facecolors="none",
                   edgecolors="limegreen",
                   linewidths=2,
                   label="Raw neighbors")

        if show_projected_neighbors:

            nn = NearestNeighbors(n_neighbors=k + 1, 
                                  metric="euclidean").fit(X_2d)

            _, indices = nn.kneighbors(X_2d[query_idx:query_idx + 1])

            projected_neighbors = indices[0][indices[0] != query_idx][:k]

            ax.scatter(X_2d[projected_neighbors, 0],
                       X_2d[projected_neighbors, 1],
                       s=160,
                       facecolors="none",
                       edgecolors="orange",
                       linewidths=1.5,
                       label="2D neighbors")

            overlap = len(set(raw_neighbors) & set(projected_neighbors))

            ax.set_title(f"{method} | Overlap: {overlap}/{k}")

        else:
            ax.set_title(method)

        # Query
        ax.scatter(X_2d[query_idx, 0],
                   X_2d[query_idx, 1],
                   c="red",
                   marker="*",
                   s=250,
                   edgecolors="black",
                   label="Query",
                   zorder=5)

        ax.set_xlabel("Component 1")
        ax.set_ylabel("Component 2")
        ax.legend()

    fig.colorbar(scatter,
                 ax=axes,
                 ticks=[0,1],
                 label="Ground Truth",
                 shrink=0.8)

    plt.show()

In [9]:

def plot_knn_evolution(query_idx, feature_space, ground_truth, predictions, threshold=0.5, max_k=100):
    ground_truth = np.asarray(ground_truth)
    predictions = np.asarray(predictions)

    valid = (np.isin(ground_truth, [0, 1]) & np.isfinite(predictions))

    # Query must not vote for itself
    valid[query_idx] = False
    valid_indices = np.flatnonzero(valid)

    max_k = min(max_k, len(valid_indices))
    if max_k < 1:
        raise ValueError("No valid neighbors available.")

    fig, axes = plt.subplots(len(feature_space), 2,
                             figsize=(15, 4 * len(feature_space)),
                             squeeze=False,
                             constrained_layout=True)

    colors = {"TP": "forestgreen",
              "FP": "darkorange",
              "FN": "crimson",
              "TN": "steelblue"}

    results = {}

    for row, (method, features) in enumerate(feature_space.items()):
        # Search only among valid, non-query samples
        nn = NearestNeighbors(n_neighbors=max_k,
                              metric="euclidean").fit(features[valid_indices])

        _, indices = nn.kneighbors(features[query_idx:query_idx + 1])

        neighbor_idx = valid_indices[indices[0]]

        gt = ground_truth[neighbor_idx]
        pred_positive = predictions[neighbor_idx] >= threshold

        categories = {
            "TP": (gt == 1) & pred_positive,
            "FP": (gt == 0) & pred_positive,
            "FN": (gt == 1) & ~pred_positive,
            "TN": (gt == 0) & ~pred_positive
        }

        k = np.arange(1, max_k + 1)

        # Cumulative share of positive DenseNet predictions
        positive_share = np.cumsum(pred_positive) / k

        # Majority vote based on DenseNet predictions
        knn_prediction = positive_share > 0.5

        # Agreement with the query's DenseNet prediction
        query_positive = predictions[query_idx] >= threshold

        prediction_agreement = np.cumsum(
            pred_positive == query_positive
        ) / k

        # Save results for further analysis
        results[method] = pd.DataFrame({
            "k": k,
            "positive_share": positive_share,
            "prediction_agreement": prediction_agreement,
            "knn_prediction": knn_prediction.astype(int),
            **{
                name: np.cumsum(mask)
                for name, mask in categories.items()
            }
        })

        # Plot A: Neighborhood Prediction Agreement Evolution
        ax = axes[row, 0]

        ax.plot(
            k,
            prediction_agreement,
            label="Prediction agreement",
            color="royalblue"
        )

        ax.axhline(
            0.5,
            color="gray",
            linestyle=":",
            label="50% agreement"
        )

        ax.set(
            title=f"{method} - Prediction Agreement Evolution",
            xlabel="Number of neighbors (k)",
            ylabel="Neighbors with same prediction (share)",
            ylim=(-0.05, 1.05)
        )

        ax.legend(fontsize=8)
        ax.grid(alpha=0.2)

        # Plot B: cumulative neighbor composition
        ax = axes[row, 1]
        bottom = np.zeros(max_k)

        for name, mask in categories.items():
            counts = np.cumsum(mask)
            ax.fill_between(
                k, bottom, bottom + counts,
                color=colors[name],
                alpha=0.8,
                label=name
            )
            bottom += counts

        ax.set(
            title=f"{method} - Neighbor Composition",
            xlabel="Number of neighbors (k)",
            ylabel="Cumulative neighbor count",
            xlim=(1, max_k)
        )
        ax.legend(fontsize=8)
        ax.grid(alpha=0.2)

    fig.suptitle(
        f"Neighborhood Evolution | Query {query_idx} | "
        f"GT: {ground_truth[query_idx]:.0f} | "
        f"Model Score: {predictions[query_idx]:.3f}",
        fontsize=15)

    plt.show()
    return results


In [10]:

def calculate_agreement_heatmap(
    feature_space,
    ground_truth,
    predictions,
    threshold=0.5,
    k_values=(10, 30, 100),
    batch_size=128
):
    gt = np.asarray(ground_truth)
    pred = np.asarray(predictions)

    valid = np.isin(gt, [0, 1]) & np.isfinite(pred)
    valid_idx = np.flatnonzero(valid)

    if len(valid_idx) <= max(k_values):
        raise ValueError("Not enough valid samples for max(k).")

    positive = pred >= threshold

    groups = {
        "TP": valid & (gt == 1) & positive,
        "FP": valid & (gt == 0) & positive,
        "FN": valid & (gt == 1) & ~positive,
        "TN": valid & (gt == 0) & ~positive
    }

    rows = []
    max_k = max(k_values)

    for method, features in feature_space.items():
        print(f"Processing {method}...")

        X = np.asarray(features)[valid_idx]

        nn = NearestNeighbors(
            n_neighbors=max_k + 1,
            metric="euclidean"
        ).fit(X)

        # Calculate all neighborhoods in batches
        agreement = np.empty((len(valid_idx), len(k_values)))

        for start in range(0, len(valid_idx), batch_size):
            end = min(start + batch_size, len(valid_idx))

            _, indices = nn.kneighbors(X[start:end])

            # Exclude each query from its own neighbors
            query_positions = np.arange(start, end)
            neighbor_indices = np.array([
                row[row != query_pos][:max_k]
                for row, query_pos in zip(indices, query_positions)
            ])

            # Model predictions of neighbors and query
            neighbor_pred = positive[valid_idx[neighbor_indices]]
            query_pred = positive[valid_idx[start:end], None]

            # Count neighbors with the same model prediction
            cumulative_matches = np.cumsum(
                neighbor_pred == query_pred,
                axis=1
            )

            for j, k in enumerate(k_values):
                agreement[start:end, j] = (
                    cumulative_matches[:, k - 1] / k
                )

        for classification, mask in groups.items():
            group_mask = mask[valid_idx]

            for j, k in enumerate(k_values):
                rows.append({
                    "Method": method,
                    "Classification": classification,
                    "k": k,
                    "Agreement": (
                        agreement[group_mask, j].mean()
                        if group_mask.any() else np.nan
                    ),
                    "N": group_mask.sum()
                })

    global_positive_rate = positive[valid].mean()
    global_negative_rate = 1 - global_positive_rate

    print(f"Predicted positive: {global_positive_rate:.2%}")
    print(f"Predicted negative: {global_negative_rate:.2%}")

    return pd.DataFrame(rows)


In [11]:

def plot_agreement_heatmaps(results):
    k_values = sorted(results["k"].unique())
    methods = list(results["Method"].unique())

    labels = {
        "TP": "TP (Pred: +)",
        "FP": "FP (Pred: +)",
        "FN": "FN (Pred: -)",
        "TN": "TN (Pred: -)"
    }

    fig, axes = plt.subplots(
        1, len(k_values),
        figsize=(5 * len(k_values) + 2, 5),
        squeeze=False,
        constrained_layout=True
    )

    for ax, k in zip(axes.flat, k_values):
        matrix = (
            results[results["k"] == k]
            .pivot(
                index="Classification",
                columns="Method",
                values="Agreement"
            )
            .reindex(
                index=list(labels.keys()),
                columns=methods
            )
        )

        matrix.index = [labels[c] for c in matrix.index]

        sns.heatmap(
            matrix,
            ax=ax,
            annot=True,
            fmt=".1%",
            cmap="RdYlGn",
            vmin=0,
            vmax=1,
            cbar=(k == k_values[-1]),
            cbar_kws={
                "label": "Neighbors with same prediction"
            },
            square=True
        )

        ax.set_title(f"k = {k}")
        ax.set_xlabel("Feature Space")
        ax.set_ylabel("Query Classification" if k == k_values[0] else "")

    fig.suptitle(
        "Neighborhood Prediction Agreement",
        fontsize=15
    )

    fig.supxlabel(
        "Percentage of k nearest neighbors classified the same as the query "
        "(TP/FP: positive, FN/TN: negative)",
        fontsize=10
    )

    plt.show()


In [12]:
import ipywidgets as widgets
from IPython.display import display, clear_output

pathology_dropdown = widgets.Dropdown(
    options=list(DATASET_LABEL_INDICES_MAP.keys()),
    description="Pathology:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="350px"))

classification_dropdown = widgets.Dropdown(
    options=["FP", "TP", "FN", "TN"],
    value="FP",
    description="Classification:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="250px"))

threshold_slider = widgets.FloatSlider(
    value=0.5,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Threshold:",
    readout_format=".2f",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="350px"))

create_button = widgets.Button(
    description="Create comparison",
    button_style="primary",
    icon="refresh")

output = widgets.Output()

def create_comparison(_):
    with output:
        clear_output(wait=True)

        pathology = pathology_dropdown.value
        classification = classification_dropdown.value
        threshold = threshold_slider.value

        # Dataset and model indices
        dataset_idx = DATASET_LABEL_INDICES_MAP[pathology]

        ground_truth = XRAY_LABELS_LIST[:, dataset_idx]
        predictions = MODEL_SCORE_PREDICTIONS_CONVERTED[:, dataset_idx]

        # Ignore samples without valid ground truth
        valid = np.isin(ground_truth, [0, 1]) & np.isfinite(predictions)
        positive = predictions >= threshold

        masks = {
            "TP": valid & (ground_truth == 1) & positive,
            "FP": valid & (ground_truth == 0) & positive,
            "TN": valid & (ground_truth == 0) & ~positive,
            "FN": valid & (ground_truth == 1) & ~positive}

        error_idx = np.flatnonzero(masks[classification])

        if len(error_idx) == 0:
            print(f"No {classification} images found for {pathology}.")
            return

        print(f"{pathology} | {classification} with treshold {threshold:.2f}| {len(error_idx)} candidates")

        # Calculate neighbors and projections
        query_idx, neighbors, feature_space = query_error_with_downprojection(
            images=images,
            error_idx=error_idx)

        # Plot 1: Nearest-neighbor image comparison
        plot_neighbor_comparison(
            query_idx=query_idx,
            neighbors=neighbors,
            images=images,
            ground_truth=ground_truth,
            predictions=predictions,
            threshold=threshold)

        # Plot 1.2: Visualize Overlap
        visualize_overlap(neighbors=neighbors)

        # Plot 2: Neighborhood projections
        plot_neighborhood_projections(
            query_idx=query_idx,
            neighbors=neighbors,
            feature_space=feature_space,
            ground_truth=ground_truth,
            k=10)

        # Plot 3: kNN evolution and neighbor composition
        plot_knn_evolution(
            query_idx=query_idx,
            feature_space=feature_space,
            ground_truth=ground_truth,
            predictions=predictions,
            threshold=threshold,
            max_k=100)

        # Plot 4: Neighborhood Agreement Heatmap
        agreement_results = calculate_agreement_heatmap(
            feature_space=feature_space,
            ground_truth=ground_truth,
            predictions=predictions,
            threshold=threshold,
            k_values=[10, 30, 100])

        plot_agreement_heatmaps(agreement_results)

        # Class distribution for selected pathology
        print(f"\nClass distribution for {pathology}:")
        print(f"Positive: {np.mean(ground_truth[valid] == 1):.2%}")
        print(f"Negative: {np.mean(ground_truth[valid] == 0):.2%}")

create_button.on_click(create_comparison)

display(
    widgets.VBox([
        pathology_dropdown,
        classification_dropdown,
        threshold_slider,
        create_button,
        output
    ]))

## Data Augmentation

How does the model respond to data augmentations with regards to

1. Brightness
2. Contrast
3. Gamma
4. Blur
5. Resolution

In [13]:
import torch
import torchvision.transforms.functional as TF

In [14]:
def augment_xray(img, method, strength=0.2):
    x = torch.as_tensor(img, dtype=torch.float32).clone()

    # TODO: Check pixel intensities
    x = ((x + 1024) / 2048).clamp(0, 1)

    match method:
        case  "contrast":
            x = TF.adjust_contrast(x, 1 + strength)
        case "gamma":
            x = TF.adjust_gamma(x, 1 + strength)
        case "blur":
            x = TF.gaussian_blur(x, kernel_size=5, sigma=max(0.1, strength*5))
        case "resolution":
            h, w = x.shape[-2:]
            scale = max(0.1, 1 - strength)
            small = TF.resize(x, [max(1, int(h*scale)), max(1, int(w*scale))], antialias=True)
            x = TF.resize(small, [h,w], antialias=True)

    return (x.clamp(0, 1) * 2048 - 1024).numpy()


In [15]:
def augment_data(augmentations, img, strength=0.2):

    augmented_xrays = {}
    for a in augmentations:
        augmented_xrays[a] = augment_xray(img, a, strength)

    return augmented_xrays


In [16]:
def predict_augmented(augmented_xrays, model, pathology):

    methods = list(augmented_xrays.keys())
    xrays = torch.as_tensor(np.stack([augmented_xrays[m] for m in methods]), dtype=torch.float32)

    device = next(model.parameters()).device
    xrays = xrays.to(device)

    model.eval()
    with torch.inference_mode():
        predictions = model(xrays)

    predictions = predictions.detach().cpu().numpy()

    return dict(zip(methods, predictions[:, pathology]))

In [17]:
def plot_augmentation_comparison(img_idx, img, augmented_xrays, old_prediction, new_predictions, ground_truth, pathology):
    n_cols = len(augmented_xrays) + 1

    fig, axes = plt.subplots(1, n_cols, figsize=(3.5 * n_cols, 4))

    images_to_plot = {"Original": img, **augmented_xrays}

    for ax, (method, image) in zip(axes, images_to_plot.items()):
        ax.imshow(np.asarray(image).squeeze(),
                  cmap="gray",
                  vmin=-1024,
                  vmax=1024)

        new_pred = (
            old_prediction if method == "Original"
            else new_predictions[method])

        ax.set_title(f"{method.capitalize()}\n"
                     f"GT: {ground_truth:.0f}\n"
                     f"Old Pred: {old_prediction:.3f}\n"
                     f"New Pred: {new_pred:.3f}")

        ax.axis("off")

    fig.suptitle(f"Data Augmentation | {pathology} | Image {img_idx}", fontsize=14)

    fig.tight_layout(rect=[0, 0, 1, 0.85])
    plt.show()

In [18]:
#augmentation_methods = ["contrast", "gamma", "blur", "resolution"]

def create_augmentation_widget(images, augmentation_methods):
    img_idx_input = widgets.BoundedIntText(value=0,
                                        min=0,
                                        max=len(images) - 1,
                                        description="Image Index:",
                                        style={"description_width": "110px"})

    aug_pathology_dropdown = widgets.Dropdown(options=list(DATASET_LABEL_INDICES_MAP.keys()),
                                            description="Pathology:",
                                            style={"description_width": "110px"},
                                            layout=widgets.Layout(width="350px"))

    strength_slider = widgets.FloatSlider(value=0.2,
                                        min=0.05,
                                        max=0.8,
                                        step=0.05,
                                        description="Strength:",
                                        style={"description_width": "110px"},
                                        layout=widgets.Layout(width="350px"))

    augment_button = widgets.Button(description="Create augmentations",
                                    button_style="primary",
                                    icon="refresh")

    augmentation_output = widgets.Output()


    def create_augmentations(_):
        with augmentation_output:
            clear_output(wait=True)

            img_idx = img_idx_input.value
            pathology = aug_pathology_dropdown.value
            strength = strength_slider.value

            dataset_idx = DATASET_LABEL_INDICES_MAP[pathology]
            model_idx = DENSENET121_RES244_CHEX_MODEL.pathologies.index(pathology)

            img = images[img_idx]
            ground_truth = XRAY_LABELS_LIST[img_idx, dataset_idx]
            old_prediction = MODEL_SCORE_PREDICTIONS_CONVERTED[img_idx, model_idx]

            augmented_xrays = augment_data(augmentation_methods, img, strength=strength)

            new_predictions = predict_augmented(augmented_xrays, DENSENET121_RES244_CHEX_MODEL, model_idx)

            plot_augmentation_comparison(img_idx=img_idx,
                                        img=img,
                                        augmented_xrays=augmented_xrays,
                                        old_prediction=old_prediction,
                                        new_predictions=new_predictions,
                                        ground_truth=ground_truth,
                                        pathology=pathology)


    augment_button.on_click(create_augmentations)

    widget = widgets.VBox([img_idx_input,
                        aug_pathology_dropdown,
                        strength_slider,
                        augment_button,
                        augmentation_output])
    return widget

In [19]:
class AugmentedXRayDataset(Dataset):
    def __init__(self, images, method=None, strength=0.2):
        self.images = images
        self.method = method
        self.strength = strength

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]

        if self.method is not None:
            img = augment_xray(img, self.method, self.strength)
    
        return torch.as_tensor(img, dtype=torch.float32)

In [20]:
# datasets = {method or "Original": AugmentedXRayDataset(images, method=method, strength=0.6) for method in augmentation_methods}

In [21]:
def process_augmented_dataset(dataset, model, batch_size=32):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    n = len(dataset)
    offset = 0

    all_predictions = []
    features = np.empty((n, *dataset[0].shape), dtype=np.float32)

    for batch in loader:
        y = model(batch)

        b = len(batch)

        features[offset:offset + b] = batch.cpu().numpy()
        all_predictions.append(y.detach().cpu())

        offset += b

    return features, np.concatenate(all_predictions, axis=0)

In [22]:
def compare_augmentations(datsets, 
                          projection_method, 
                          visualization_fns, 
                          visualization_names,
                          ground_truth, 
                          dataset_label_indices_map,
                          model=DENSENET121_RES244_CHEX_MODEL,
                          dataset_gender_list=XRAY_GENDER_LIST,
                          dataset_ages=XRAY_AGE_LIST,
                          pathologies=DATASET_PATHOLOGIES,
                          threshold=0.7):

    results = {}

    for method, dataset in datasets.items():
        print(f"Processing: {method}")

        features, predictions = process_augmented_dataset(dataset, model)

        downprojector = Downprojector(features.reshape(-1, 224* 224))

        #fig = create_multi_pathology_viewer(downprojector=downprojector,
        #                                    projection_method=projection_method,
        #                                    visualization_fns=visualization_fns,
        #                                    visualization_names=visualization_names,
        #                                    model_score_predictions=predictions,
        #                                    ground_truth=ground_truth,
        #                                    dataset_label_indices_map=dataset_label_indices_map,
        #                                    dataset_gender_list=dataset_gender_list,
        #                                    dataset_ages=dataset_ages,
        #                                    threshold=threshold)

        results[method] = {"features": features, "predictions": predictions} #"figure":fig}

    return results

In [23]:
## !!! RESSOURCE EXTENSIVE - DO NOT RUN THIS UNLESS NECESSARY -> load from joblib dump instead
#comparison = compare_augmentations(datasets, "S-UMAP", visualization_functions, visualization_names, XRAY_LABELS_LIST, DATASET_LABEL_INDICES_MAP)

In [24]:
#import joblib

#joblib.dump(comparison, "comparison_s_umap.joblib", compress=0)
#comparison = joblib.load("comparison_s_umap.joblib")

In [25]:
def compare_prediction_scores(results, pathologies, model_pathologies):
    original = results["Original"]["predictions"]
    rows = []

    for method, result in results.items():
        if method == "Original":
            continue

        predictions = result["predictions"]

        for pathology in pathologies:
            model_idx = model_pathologies.index(pathology)

            delta = predictions[:, model_idx] - original[:, model_idx]

            rows.append({
                "Pathology": pathology,
                "Augmentation": method,
                "Mean Original": np.nanmean(original[:, model_idx]),
                "Mean Augmented": np.nanmean(predictions[:, model_idx]),
                "Mean Delta": np.nanmean(delta),
                "Mean Absolute Delta": np.nanmean(np.abs(delta)),
                "Max Absolute Delta": np.nanmax(np.abs(delta))})

    return pd.DataFrame(rows)

In [26]:
#score_comparison = compare_prediction_scores(comparison, DATASET_PATHOLOGIES, DENSENET121_RES244_CHEX_MODEL.pathologies)

#display(score_comparison)

In [27]:
def plot_prediction_sensitivity(score_comparison):

    pivot = score_comparison.pivot(index="Pathology",
                                columns="Augmentation",
                                values="Mean Absolute Delta")

    fig, ax = plt.subplots(figsize=(9, 7))

    im = ax.imshow(pivot.values, cmap="RdBu_r", aspect="auto")

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)

    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)

    fig.colorbar(im, ax=ax, label="Mean Absolute Prediction Change")

    ax.set_title("Prediction Sensitivity Across Pathologies")
    plt.tight_layout()
    plt.show()

In [28]:
def get_classification_counts(gt, pred, threshold=0.7):
    valid = np.isfinite(gt) & np.isfinite(pred)

    gt = gt[valid] == 1
    pred = pred[valid] >= threshold

    return {
        "TP": np.sum(gt & pred),
        "FP": np.sum(~gt & pred),
        "FN": np.sum(gt & ~pred),
        "TN": np.sum(~gt & ~pred)
    }


def calculate_augmentation_changes(
    results,
    ground_truth,
    dataset_label_indices_map,
    model=DENSENET121_RES244_CHEX_MODEL,
    threshold=0.7,
    baseline="Original"
):
    rows = []
    ground_truth = np.asarray(ground_truth)

    for method, result in results.items():
        predictions = np.asarray(result["predictions"])

        if len(predictions) != len(ground_truth):
            raise ValueError(f"Dataset size mismatch for {method}")

        for pathology, gt_idx in dataset_label_indices_map.items():
            if pathology not in model.pathologies:
                continue

            pred_idx = model.pathologies.index(pathology)

            counts = get_classification_counts(
                ground_truth[:, gt_idx],
                predictions[:, pred_idx],
                threshold
            )

            rows.append({
                "Augmentation": method,
                "Pathology": pathology,
                **counts
            })

    df = pd.DataFrame(rows).set_index(
        ["Pathology", "Augmentation"]
    )

    baseline_counts = df.xs(baseline, level="Augmentation")

    delta = df.copy()

    for pathology in baseline_counts.index:
        mask = df.index.get_level_values("Pathology") == pathology

        delta.loc[mask, :] = (
            df.loc[mask, :].to_numpy()
            - baseline_counts.loc[pathology].to_numpy()
        )

    delta = delta[
        delta.index.get_level_values("Augmentation") != baseline
    ]

    return df, delta

In [29]:
#counts, delta = calculate_augmentation_changes(comparison, XRAY_LABELS_LIST, dataset_label_indices_map=DATASET_LABEL_INDICES_MAP)

In [30]:
from matplotlib.colors import TwoSlopeNorm

def plot_augmentation_heatmaps(delta):
    classifications = ["TP", "FP", "FN", "TN"]

    fig, axes = plt.subplots(2, 2, figsize=(15, 10), constrained_layout=True)

    fig.suptitle("Impact of Data Augmentation on Classification Outcomes", fontsize=16, fontweight="bold")

    for ax, classification in zip(axes.flat, classifications):
        matrix = delta[classification].unstack("Augmentation")
        values = matrix.to_numpy(dtype=float)

        max_abs = max(np.nanmax(np.abs(values)), 1)

        im = ax.imshow(values,
                        cmap="RdBu" if classification in ["TP", "TN"] else "RdBu_r",
                        norm=TwoSlopeNorm(vmin=-max_abs, vcenter=0, vmax=max_abs),
                        aspect="auto")

        ax.set_xticks(range(len(matrix.columns)))
        ax.set_xticklabels(matrix.columns, rotation=45, ha="right")

        ax.set_yticks(range(len(matrix.index)))
        ax.set_yticklabels(matrix.index)

        ax.set_title(f"Δ {classification}")

        for i in range(values.shape[0]):
            for j in range(values.shape[1]):
                if np.isfinite(values[i, j]):
                    ax.text(j, i,
                        f"{values[i, j]:+.0f}",
                        ha="center",
                        va="center",
                        fontsize=8)

        fig.colorbar(im, ax=ax, shrink=0.8)

    plt.show()
    #plt.close(fig)

In [31]:
# plot_augmentation_heatmaps(delta)

In [32]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.colors import qualitative
from IPython.utils.capture import capture_output


def plot_augmentation_dashboard(
    comparison,
    ground_truth,
    dataset_label_indices_map,
    delta,
    model=DENSENET121_RES244_CHEX_MODEL,
    baseline="Original",
    log_y=False,
    y_min=0.001,
):
    ground_truth = np.asarray(ground_truth)
    classifications = ["TP", "FP", "FN", "TN"]

    pathologies = [
        p for p in dataset_label_indices_map
        if p in model.pathologies
        and p in delta.index.get_level_values("Pathology")
    ]

    methods = list(comparison)
    scores = {
        method: np.asarray(comparison[method]["predictions"])
        for method in methods
    }

    nrows = len(pathologies)
    row_height = 300
    row_gap = 130
    top_margin = 150
    bottom_margin = 80

    plot_height = nrows * row_height + (nrows - 1) * row_gap

    fig = make_subplots(
        rows=nrows,
        cols=3,
        column_widths=[0.36, 0.32, 0.32],
        subplot_titles=[
            title
            for p in pathologies
            for title in (
                f"{p} — Δ counts",
                f"{p} — ROC",
                f"{p} — Precision–Recall",
            )
        ],
        horizontal_spacing=0.07,
        vertical_spacing=row_gap / plot_height if nrows > 1 else 0,
    )

    # A shared color scale makes count changes comparable across pathologies.
    max_abs = max(
        float(np.nanmax(np.abs(delta[classifications].to_numpy()))),
        1,
    )

    def visible_y(values):
        values = np.asarray(values, dtype=float)
        return [
            float(v) if np.isfinite(v) and (not log_y or v > 0)
            else None
            for v in values
        ]

    for row, pathology in enumerate(pathologies, start=1):
        # Left: the same delta data as plot_augmentation_heatmaps,
        # arranged as four classification rows for this pathology.
        changes = delta.xs(pathology, level="Pathology")
        augmentations = [m for m in methods if m in changes.index]
        matrix = changes.reindex(augmentations)[classifications].T

        fig.add_trace(
            go.Heatmap(
                z=matrix.to_numpy(dtype=float),
                x=list(matrix.columns),
                y=list(matrix.index),
                coloraxis="coloraxis",
                texttemplate="%{z:+.0f}",
                hovertemplate=(
                    "%{y}<br>%{x}<br>Change: %{z:+.0f}"
                    "<extra></extra>"
                ),
            ),
            row=row, col=1,
        )
        fig.update_yaxes(autorange="reversed", row=row, col=1)
        fig.update_xaxes(tickangle=-40, row=row, col=1)

        # Use the same valid samples for every augmentation's curves.
        gt_idx = dataset_label_indices_map[pathology]
        pred_idx = model.pathologies.index(pathology)
        true = ground_truth[:, gt_idx]

        valid = np.isin(true, [0, 1])
        for method in methods:
            valid &= np.isfinite(scores[method][:, pred_idx])

        has_both_classes = np.unique(true[valid]).size == 2

        if has_both_classes:
            for j, method in enumerate(methods):
                with capture_output():
                    evaluation = evaluate_multilabel_model(
                        y_true=true[valid, None],
                        y_scores=scores[method][valid, pred_idx][:, None],
                        pathology_names=[pathology],
                    )

                metrics = evaluation[pathology]
                line = dict(
                    color=qualitative.Plotly[j % len(qualitative.Plotly)],
                    width=3 if method == baseline else 2,
                    dash="dash" if method == baseline else "solid",
                )

                for col, x, y, summary in [
                    (
                        2,
                        metrics["fpr"],
                        metrics["tpr"],
                        f"AUC: {metrics['roc_auc']:.3f}",
                    ),
                    (
                        3,
                        metrics["recall"],
                        metrics["precision"],
                        f"AP: {metrics['average_precision']:.3f}",
                    ),
                ]:
                    fig.add_trace(
                        go.Scatter(
                            x=x,
                            y=visible_y(y),
                            mode="lines",
                            name=method,
                            legendgroup=method,
                            showlegend=False,
                            line=line,
                            connectgaps=False,
                            hovertemplate=(
                                f"{method}<br>{summary}"
                                "<br>x: %{x:.3f}<br>y: %{y:.3f}"
                                "<extra></extra>"
                            ),
                        ),
                        row=row, col=col,
                    )

            # ROC chance line; start above zero for logarithmic axes.
            start = y_min if log_y else 0
            chance = np.linspace(start, 1, 200)
            fig.add_trace(
                go.Scatter(
                    x=chance,
                    y=chance,
                    mode="lines",
                    line=dict(color="gray", dash="dot"),
                    showlegend=False,
                    hoverinfo="skip",
                ),
                row=row, col=2,
            )

            fig.add_hline(
                y=float(true[valid].mean()),
                line_color="gray",
                line_dash="dot",
                row=row, col=3,
            )

        for col, xlabel, ylabel in [
            (2, "False Positive Rate", "True Positive Rate"),
            (3, "Recall", "Precision"),
        ]:
            fig.update_xaxes(
                title=dict(text=xlabel, standoff=10),
                range=[0, 1],
                automargin=True,
                row=row, col=col,
            )
            fig.update_yaxes(
                title=dict(text=ylabel, standoff=10),
                type="linear",
                range=[0, 1.02],
                tickformat=".1f",
                dtick=0.2,
                automargin=True,
                row=row, col=col,
            )

    # Show each augmentation once in the legend.
    seen = set()
    for trace in fig.data:
        if trace.legendgroup and trace.legendgroup not in seen:
            trace.showlegend = True
            seen.add(trace.legendgroup)

    fig.update_layout(
        title=dict(
            text="Augmentation comparison by pathology",
            y=1,
            yref="container",
            yanchor="top",
            pad=dict(t=20),
        ),
        template="plotly_white",
        height=plot_height + top_margin + bottom_margin,
        autosize=True,
        margin=dict(
            t=top_margin,
            b=bottom_margin,
            l=80,
            r=100,
        ),
        coloraxis=dict(
            colorscale="RdBu",
            reversescale=True,
            cmin=-max_abs,
            cmax=max_abs,
            colorbar=dict(title="Δ count", len=0.25),
        ),
        legend=dict(
            title="Augmentation",
            orientation="h",
            y=1.015,
            yanchor="bottom",
            x=0,
            groupclick="togglegroup",
        ),
    )

    fig.show(config={"responsive": True})
    return fig